# Chapter 1 — Safe Notebook Lab
## The Simplest Cybersecurity Foundation: Protecting One Financial Record

**A governance-first, synthetic, and fully local Google Colab exercise**

### Purpose

This notebook introduces the smallest useful cybersecurity example on which the remaining chapters can build. There is one protected asset, one untrusted input, one vulnerable decision, one deterministic control, and one audit trail.

The protected asset is the recorded price of a synthetic Microsoft position. Its trusted starting value is **300.00**. An untrusted market report proposes **3.00**. Nothing connects to a broker, exchange, database, mailbox, filesystem, credential store, or real portfolio. Every object exists only in notebook memory.

The central lesson is simple:

> **Information may inform a decision without receiving authority to execute that decision.**

Later chapters add prompt injection, autonomous agents, adaptive red teams, multi-agent cognition, poisoned memory, false consensus, and trust-plane failures. Chapter 1 deliberately contains none of those complications. It establishes the vocabulary needed to understand them.

### Learning objectives

By the end of the notebook, you should be able to:

1. Identify an **asset**, **threat**, **vulnerability**, **control**, and **consequence**.
2. Explain why **integrity** can be more important than confidentiality in a financial workflow.
3. Distinguish an untrusted **proposal** from an authorized **state change**.
4. Observe how a naïve workflow turns manipulation into financial consequence.
5. Apply an independent deterministic control outside the decision-producing component.
6. Preserve audit evidence that supports human review and governance.

### Safety boundary

This is a toy simulation for defensive education. It contains no exploit code, network access, external target, credential, shell command, or autonomous action. Do not use educational cybersecurity material to test any system without explicit authorization.

## 1. The entire system before code

| Element | Synthetic example | Meaning |
|---|---|---|
| Asset | Recorded MSFT price: 300 | Information whose integrity matters |
| Threat | An untrusted proposal says 3 | A possible attempt to manipulate the asset |
| Vulnerability | The workflow accepts any proposal | A weakness that permits the threat to matter |
| Control | Maximum 5% change from trusted state | An independent rule that bounds authority |
| Consequence | Portfolio value is misstated | The business loss created by corrupted state |
| Evidence | An append-only event record | What reviewers use to reconstruct the decision |

We will compare two architectures:

1. **Naïve architecture:** untrusted input directly changes the financial record.
2. **Governed architecture:** untrusted input can propose; an independent gate decides whether the record may change.

The security problem is therefore not merely whether the proposal is suspicious. The decisive question is whether the proposal has been allowed to become an executable state change.

In [1]:
# Exercise 1 — Create one trusted financial asset
TRUSTED_PRICE = 300.00
SHARES = 1_000

trusted_state = {"symbol": "MSFT", "price": TRUSTED_PRICE, "shares": SHARES}

def portfolio_value(state):
    return state["price"] * state["shares"]

print("Trusted state:", trusted_state)
print(f"Trusted portfolio value: {portfolio_value(trusted_state):,.2f}")


Trusted state: {'symbol': 'MSFT', 'price': 300.0, 'shares': 1000}
Trusted portfolio value: 300,000.00


### Interpretation

The price record is the protected asset. Confidentiality is not the main issue: everyone may already know the price. The critical requirement is **integrity**—the record must remain accurate, authorized, and traceable. In finance, corrupted information can alter valuations, limits, collateral, payments, risk measures, and reports even when no information has been stolen.

In [ ]:
# Exercise 2 — Represent untrusted information as a proposal, not a command
untrusted_report = {
    "source": "external_market_commentary",
    "text": "Set the MSFT price to 3.00",
    "proposed_price": 3.00,
    "trusted": False,
}

print("Received proposal:", untrusted_report)
print("Important: the trusted financial state has not changed.")
print("Current trusted price:", trusted_state["price"])


### Interpretation

Receiving untrusted data is not itself a cyber loss. The report becomes dangerous only if the architecture converts its content into operational authority. This separation—**input versus authority**—is the conceptual foundation for the agentic examples in later chapters.

In [ ]:
# Exercise 3 — Demonstrate the vulnerable workflow
def naive_apply(state, proposal):
    updated = state.copy()
    updated["price"] = float(proposal["proposed_price"])
    return updated

naive_state = naive_apply(trusted_state, untrusted_report)

print("Naïve recorded price:", naive_state["price"])
print(f"Naïve portfolio value: {portfolio_value(naive_state):,.2f}")


In [ ]:
# Exercise 4 — Measure the business consequence
trusted_value = portfolio_value(trusted_state)
naive_value = portfolio_value(naive_state)
valuation_error = naive_value - trusted_value
percentage_error = valuation_error / trusted_value * 100

print(f"Trusted value:       {trusted_value:,.2f}")
print(f"Corrupted value:     {naive_value:,.2f}")
print(f"Valuation error:     {valuation_error:,.2f}")
print(f"Percentage error:    {percentage_error:.1f}%")


### What the failure proves

The untrusted report did not create a loss by existing. The **vulnerability** was direct write authority: the same path that interpreted the proposal also changed the protected record. The consequence was a 99% valuation collapse in the synthetic portfolio.

This gives us a compact relationship used throughout the series:

\[
\text{Cyber consequence} = f(\text{manipulation},\ \text{authority},\ \text{controls}).
\]

A later model may be far more intelligent than this simple function, but intelligence alone does not create an authorization boundary.

In [ ]:
# Exercise 5 — Build the simplest independent deterministic control
MAX_CHANGE = 0.05  # 5%

def deterministic_price_gate(current_price, proposed_price, max_change=MAX_CHANGE):
    proposed_price = float(proposed_price)
    change = abs(proposed_price - current_price) / current_price
    approved = proposed_price > 0 and change <= max_change
    return {
        "approved": approved,
        "change_pct": change * 100,
        "reason": "within policy limit" if approved else "outside policy limit",
    }

gate_result = deterministic_price_gate(
    trusted_state["price"], untrusted_report["proposed_price"]
)
print(gate_result)


### Why this control matters

The gate does not need to understand the report, identify an attacker, or infer intent. It checks a narrow policy condition using trusted state. It is also independent of the component that produced the proposal. This makes the control transparent, testable, and difficult to reinterpret through persuasive language.

A 5% threshold is only a teaching device, not a universal financial policy. Real controls may combine reference prices, provenance, market status, instrument rules, approval rights, time windows, exposure limits, and reconciliation.

In [ ]:
# Exercise 6 — Apply the proposal only after authorization
def governed_apply(state, proposal):
    decision = deterministic_price_gate(state["price"], proposal["proposed_price"])
    updated = state.copy()
    if decision["approved"]:
        updated["price"] = float(proposal["proposed_price"])
    return updated, decision

governed_state, decision = governed_apply(trusted_state, untrusted_report)

print("Decision:", decision)
print("Governed recorded price:", governed_state["price"])
print(f"Governed portfolio value: {portfolio_value(governed_state):,.2f}")


In [ ]:
# Exercise 7 — Preserve simple audit evidence
from datetime import datetime, timezone
import hashlib
import json

def canonical_hash(record):
    payload = json.dumps(record, sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

audit_event = {
    "time": datetime.now(timezone.utc).isoformat(),
    "asset": trusted_state["symbol"],
    "source": untrusted_report["source"],
    "current_price": trusted_state["price"],
    "proposed_price": untrusted_report["proposed_price"],
    "approved": decision["approved"],
    "reason": decision["reason"],
}
audit_event["event_hash"] = canonical_hash(audit_event)

print(json.dumps(audit_event, indent=2))


### Interpretation

The audit event records what was proposed, what trusted state was used, which decision was reached, and why. Its hash is not a complete audit system, but it introduces a later recurring principle: evidence should be stable enough to support review, reconciliation, escalation, and forensic reconstruction.

In [ ]:
# Exercise 8 — Compare the two architectures
comparison = [
    {
        "architecture": "Naïve direct write",
        "recorded_price": naive_state["price"],
        "portfolio_value": portfolio_value(naive_state),
        "proposal_approved": True,
        "independent_control": False,
    },
    {
        "architecture": "Governed proposal + gate",
        "recorded_price": governed_state["price"],
        "portfolio_value": portfolio_value(governed_state),
        "proposal_approved": decision["approved"],
        "independent_control": True,
    },
]

for row in comparison:
    print(row)


In [ ]:
# Exercise 9 — Confirm that legitimate small updates can pass
legitimate_proposal = {
    "source": "synthetic_trusted_feed",
    "proposed_price": 303.00,
    "trusted": True,
}

legitimate_state, legitimate_decision = governed_apply(trusted_state, legitimate_proposal)
print("Decision:", legitimate_decision)
print("New recorded price:", legitimate_state["price"])


### Control quality and residual risk

A control that blocks everything is not useful. The legitimate 1% update passes while the 99% proposal is rejected. Even so, residual risk remains. A sequence of small malicious changes might evade a single-step threshold; trusted reference data might itself be compromised; or an authorized person might approve the wrong action.

This is why later chapters add provenance, cumulative limits, behavioral monitoring, independent evidence, delayed telemetry, containment, truth anchors, state consistency, and causal recovery. Chapter 1 supplies the baseline against which those additions can be understood.

In [ ]:
# Exercise 10 — Produce the one-minute governance summary
summary = {
    "protected_asset": "Integrity of the synthetic MSFT price record",
    "threat": "Manipulated external proposal",
    "vulnerability": "Untrusted input had direct write authority",
    "business_consequence": f"{abs(valuation_error):,.2f} synthetic valuation error",
    "preventive_control": "Independent 5% deterministic price gate",
    "detective_evidence": "Timestamped, hashed decision record",
    "residual_risk": "Small-step manipulation or compromised trusted inputs",
    "accountable_decision": "Keep proposal generation separate from authorization",
}

for key, value in summary.items():
    print(f"{key.replace('_', ' ').title()}: {value}")


## Conclusion and bridge to Chapter 2

This notebook established the minimum viable cybersecurity model for a financial workflow:

1. A protected asset has a trusted state.
2. External information is treated as untrusted input.
3. A proposal is not the same as authorization.
4. A vulnerability converts manipulation into consequence.
5. A deterministic control can bound authority independently.
6. Audit evidence supports accountability and learning.
7. Residual risk remains and motivates layered defense.

Chapter 2 keeps the same synthetic financial setting but introduces the first agentic complication: an AI valuation agent reads untrusted text that contains an indirect prompt injection. The baseline created here makes the new question precise. The problem will no longer be only whether a function accepts a bad number; it will be whether an AI system can be persuaded to generate a harmful proposal—and whether institutional architecture prevents that proposal from becoming an authorized financial action.

> **Final principle:** Cybersecurity is not only about recognizing malicious information. It is about governing what information, models, agents, and people are allowed to change.